### Evals for AI Engineers - Chap 2 LLMs and Eval Basics
- uv add litellm, phonenumbers


#### Example: Email Summarizer

In [19]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from IPython.display import Markdown, display
import phonenumbers

from litellm import embedding, completion

In [25]:
email_text = """
From: Alice Johnson <alice.johnson@example.com>
To: Support Team

Hi,

Could you please reset my account password? I would also appreciate it if you could verify that my email address on file is correct.
I would also prefer to be contacted by phone. Call me at (415) 555-9021 ext. 7.

Thank you,
Alice Johnson
"""


#### A single LLM call

In [8]:
response = completion(model="openai/gpt-4.1", 
                      messages=[
                          {"role": "system","content": "You are an assistant that summarizes emails."},
                          {"role": "user","content": "Extract the sender's name and summarize the key requests:\n\n" + email_text}
                      ])
reply = response.choices[0].message.content
display(Markdown(reply))

Sender's Name: Alice Johnson

Summary of Key Requests:
- Requests a password reset for her account.
- Asks to verify that her email address on file is correct.
- Prefers to be contacted by phone.

#### A conversation

In [17]:
messages = [
    {
        "role": "system",
        "content": (
            "You are an assistant that summarizes and "
            "answers questions about emails."
        ),
    },
    {"role": "user", "content": "Summarize this email:\n\n" + email_text},
]

response1 = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
)
print(response1["choices"][0]["message"]["content"])

# Add the model's reply to the conversation and continue
messages.append({
    "role": "assistant",
    "content": response1["choices"][0]["message"]["content"],
})
messages.append({
    "role": "user",
    "content": "Does she mention her preferred contact method?",
})

response2 = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
)

display(Markdown(response2["choices"][0]["message"]["content"]))


Alice Johnson is requesting the Support Team to reset her account password and verify that the correct email address is on file for her account. She prefers to be contacted by phone for further communication.


Yes, Alice Johnson mentions that she prefers to be contacted by phone.

#### Retrieval

In [15]:
# Small archive of past support emails
past_emails = [
    "Ticket #4521: Please reset my password. I cannot log in.",
    "Ticket #4605: Could you verify my email address on file?",
    "Ticket #4710: I need help updating my phone number.",
]

In [14]:
# Cosine similarity scores two embedding vectors from -1 (opposite)
# to 1 (identical). Higher means the text is more similar.
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [18]:
# Embedding-based retrieval
corpus_response = embedding(
    model="text-embedding-3-small",
    input=past_emails,
)
corpus_embeddings = [item["embedding"] for item in corpus_response["data"]]

query_response = embedding(
    model="text-embedding-3-small",
    input=email_text,
)
query_embedding = query_response["data"][0]["embedding"]

scores = [cosine_similarity(query_embedding, e) for e in corpus_embeddings]
retrieved = past_emails[int(np.argmax(scores))]

# Combine retrieved context with the new email
messages = [
    {
        "role": "system",
        "content": "You are an assistant that summarizes emails with context.",
    },
    {
        "role": "user",
        "content": (
            f"Email:\n{email_text}\n\n"
            f"Related past email:\n{retrieved}"
        ),
    },
]

response = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
)

display(Markdown(response["choices"][0]["message"]["content"]))

Alice Johnson has requested a password reset for her account and asked for confirmation that her email address on file is correct. She also prefers to be contacted by phone. This request is related to a previous ticket (#4605) in which she had already inquired about verifying her email address.

- Keyword-based retrieval could be swapped in here using a library like whoosh or elasticsearch, returning the top BM25 match instead of computing embeddings.

#### Tool call
- normalizing phone number

In [20]:
# Define the tool as a Python function
def normalize_phone(number: str):
    parsed = phonenumbers.parse(number, "US")
    return phonenumbers.format_number(
        parsed, phonenumbers.PhoneNumberFormat.E164
    )

In [22]:
# Tell the model what the tool is and how to call it
tools = [
    {
        "type": "function",
        "function": {
            "name": "normalize_phone",
            "description": (
                "Convert a phone number to standardized E.164 format"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "number": {"type": "string"}
                },
                "required": ["number"],
            },
        },
    }
]

In [28]:
messages = [
    {
        "role": "system",
        "content": (
            "You summarize emails. If the email contains a phone number, "
            "you MUST call the normalize_phone tool on it before writing the "
            "summary, and use the normalized number in the summary."
        ),
    },
    {
        "role": "user",
        "content": f"Summarize this email:\n\n{email_text}",
    },
]

resp = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
    tools=tools,
    # "auto" (the default) lets the model skip tools. "required" forces at least one tool call.
    # To force a specific tool: {"type": "function", "function": {"name": "normalize_phone"}}
    tool_choice="required",
)
msg = resp["choices"][0]["message"]

# When the model calls a tool, msg["content"] is usually None and the request
# lives in msg["tool_calls"] instead, so inspect both.
if msg.get("tool_calls"):
    for call in msg["tool_calls"]:
        print(f"Tool requested: {call['function']['name']}({call['function']['arguments']})")
else:
    display(Markdown(msg["content"]))

Tool requested: normalize_phone({"number":"(415) 555-9021 ext. 7"})


In [29]:


# If the model calls the tool, run it and send back the result
if msg.get("tool_calls"):
    # Append the assistant's message containing the tool calls
    messages.append(msg)
    print("Tool calls detected. Executing tool calls...")

    for call in msg["tool_calls"]:
        if call["function"]["name"] == "normalize_phone":
            args = json.loads(call["function"]["arguments"])
            result = normalize_phone(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "name": "normalize_phone",
                "content": json.dumps(result),
            })

# Continue the conversation until the model produces a final summary
resp2 = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
    tools=tools,
)
display(Markdown(resp2["choices"][0]["message"]["content"]))

Tool calls detected. Executing tool calls...


Alice Johnson requested a password reset for her account and verification of her email address. She prefers to be contacted by phone at +14155559021.

#### Prompt caching

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")